###### 03_silver_layer

###### Purpose

Purpose of this notebook is to focus on cleaning and standardization on the data that is loaded in the Bronze Delta table.


###### Technologies Used

 -  Databricks

 -  Apache Spark

 -  PySpark

 -  Delta Lake

 - Unity Catalog


###### Input

- Bronze delta table

######  Output

- Silver delta table


######  Architecture

```text

Read Bronze
      ↓
Data Quality Checks
      ↓
Clean Invalid Values
      ↓
Standardize Data Types
      ↓
Validate Dataset
      ↓
Write Silver Delta Table

```


###### Skills Covered

- Delta Lake

- Data Cleaning

- Data Quality

- Silver Layer

###### Section 0 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 1 : Import libraries

In [0]:
from pyspark.sql.functions import count, when, col, trim

###### Section 2 : Read Bronze table

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
display(bronze_df)

###### Section 3 : Inspect data quality

In [0]:
bronze_df.printSchema()

#check nulls
bronze_df.select(count(when(col("TotalCharges").isNull(), True)).alias("null_totalcharges")).show() 

###### Section 4 : Data Cleaning and Standardization

In [0]:
#clean data
silver_df = (bronze_df.withColumn("TotalCharges", when(trim(col("TotalCharges")) == "", None).otherwise(col("TotalCharges")))
    .withColumn("TotalCharges", col("TotalCharges").cast("double")))


#verify schema
silver_df.printSchema()


# check nulls
null_count = silver_df.select(count(when(col("TotalCharges").isNull(), True)).alias("null_totalcharges")).count()

# duplicate customer ids
duplicate_count = (
    silver_df
    .groupBy("customerID")
    .count()
    .filter(col("count") > 1)
    .count()
)


print("Data Quality Summary")
print("--------------------")
print(f"Null TotalCharges : {null_count}")
print(f"Duplicate Customer IDs : {duplicate_count}")
print(f"Bronze rows : {bronze_df.count()}")
print(f"Silver rows : {silver_df.count()}")

###### Section 5 : Write to the Silver delta table

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

###### Section 6 :  Validation silver dataset

In [0]:

silver_df.count()

silver_df.printSchema()

display(silver_df.limit(10))

###### Key Learnings

- Cleaned invalid and missing values.

- Converted TotalCharges to a numeric data type.

- Checked for duplicate customer IDs.

- Standardized the dataset for downstream analytics and machine learning.

###### Notebook Conclusion

- Successfully cleaned and standardized the Bronze dataset, validated data quality, and stored the processed data in the Silver Delta table for downstream feature engineering.

###### Next Notebook

04_gold_layer

Purpose

- Create feature-engineered, model-ready data by transforming the Silver dataset into the Gold Delta table.